# Question 1 — Stage 1 and Stage 2: Environment, requirements and data acquisition

Group number: 4  
Student 1: Niyonsenga Felicien  
Student 2: Leonce  
Question: Q1 — Maize yield in Rwanda  
Stage: 1–2

## Important dataset note
The supplied assignment PDF describes a file named `maize_yield_rwanda_raw.csv` with columns such as `farm_id` and `yield_kg_per_ha`. The uploaded CSV in this project is instead named `maize_yield_rwanda_rw.csv` and has 1,426 rows, 20 columns, and target `yield_t_ha`. This notebook follows the **actual uploaded CSV**. Confirm with your lecturer that this is the correct Q1 dataset before submission.

## Problem and data requirements
The aim is to prepare farm/season records so a later modelling team can predict maize yield per hectare. The target in the uploaded data is `yield_t_ha` (tonnes per hectare). Useful predictors include location, year, season, altitude, rainfall, temperature, soil, seed, farming practice, fertiliser application, plant density, farm area, irrigation and pest incidence.

We must not train a prediction model in this assignment. We must keep the raw file unchanged, document all cleaning, and avoid using information only known after harvest as a predictor.

In [3]:
# Install library version 
%pip install matplotlib seaborn pandas numpy scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Environment and library version check
import sys, pandas, numpy, matplotlib, seaborn, sklearn
print("Python:", sys.version)
print("pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", seaborn.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
pandas: 3.0.2
NumPy: 2.4.4
Matplotlib: 3.11.2
Seaborn: 0.13.2
scikit-learn: 1.9.1


## Load the supplied data
The original CSV is stored in `data/raw/` and is never edited directly.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
sns.set_theme()
CWD = Path.cwd()
raw_candidates = [
    CWD / "data" / "raw" / "maize_yield_rwanda_rw.csv",
    CWD / "q1_maize_yield" / "data" / "raw" / "maize_yield_rwanda_rw.csv",
    CWD.parent / "data" / "raw" / "maize_yield_rwanda_rw.csv",
]
RAW_PATH = next((p for p in raw_candidates if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError("Could not find data/raw/maize_yield_rwanda_rw.csv. Open the notebook from the extracted project.")
PROJECT = RAW_PATH.parents[2]
INTERIM = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(RAW_PATH)

print("Dataset shape (rows, columns):", df.shape)
display(df.head())
display(df.tail())
df.info()

Dataset shape (rows, columns): (1426, 20)


,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha
0,RW-MZ-00001,2015,A,Eastern,Bugesera,1349,NaN,22.9,NaN,Clay,RHM104,Improved,117.8,57.4,1.41,61803,0.63,No,NaN,2.71
1,RW-MZ-00002,2015,A,Eastern,Bugesera,1427,489.8,22.1,6.02,Volcanic,RHM104,Improved,38.1,62.7,3.69,52919,0.17,No,0.339,4.09
2,RW-MZ-00003,2015,A,Northern,Burera,2033,722.6,16.3,5.73,Clay loam,Local,Improved,121.1,50.7,5.18,54406,0.22,No,0.049,2.58
3,RW-MZ-00004,2015,A,Northern,Burera,2026,794.4,16.4,5.20,Clay loam,ZM621,Improved,93.7,60.4,1.68,48119,0.48,No,0.171,2.45
4,RW-MZ-00005,2015,A,Northern,Gakenke,1974,644.2,18.3,5.92,Loam,Local,Improved,84.1,70.4,0.36,56787,1.16,No,0.001,2.09


,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha
1421,RW-MZ-01422,2024,C,Southern,Nyanza,1748,228.6,19.3,5.96,Clay,ZM621,Traditional,9.1,13.6,2.86,36249,1.54,No,0.284,1.89
1422,RW-MZ-01423,2024,C,Northern,Rulindo,1881,211.5,18.4,5.45,Volcanic,ZM607,Improved,36.3,66.0,1.64,64018,0.34,No,0.403,2.21
1423,RW-MZ-01424,2024,C,Northern,Rulindo,1887,242.3,18.2,5.70,Volcanic,WH505,Traditional,7.4,37.7,1.29,24474,1.95,No,0.304,2.10
1424,RW-MZ-01425,2024,C,Western,Rusizi,1409,310.5,19.6,5.02,Sandy loam,ZM607,Traditional,13.7,0.0,0.55,43111,0.20,No,0.542,1.40
1425,RW-MZ-01426,2024,C,Western,Rusizi,1583,391.0,20.5,5.01,Sandy loam,ZM607,Improved,86.7,50.8,2.50,54197,0.77,No,0.193,2.12


<class 'pandas.DataFrame'>
RangeIndex: 1426 entries, 0 to 1425
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   record_id             1426 non-null   str    
 1   year                  1426 non-null   int64  
 2   season                1426 non-null   str    
 3   province              1426 non-null   str    
 4   district              1426 non-null   str    
 5   altitude_m            1426 non-null   int64  
 6   rainfall_mm           1415 non-null   float64
 7   avg_temp_c            1426 non-null   float64
 8   soil_ph               1412 non-null   float64
 9   soil_type             1426 non-null   str    
 10  seed_variety          1426 non-null   str    
 11  farming_practice      1426 non-null   str    
 12  npk_kg_ha             1426 non-null   float64
 13  urea_kg_ha            1426 non-null   float64
 14  manure_t_ha           1426 non-null   float64
 15  plant_density_per_ha  1426 non-n

## Data dictionary — initial interpretation

Units below follow column names and observed structure; confirm them with the dataset provider if possible.

In [6]:
dictionary = pd.DataFrame([
    ("record_id","Unique record code","Categorical","Identifier","Unique code per record"),
    ("year","Observation year","Integer","Year","2015–2024 in uploaded file"),
    ("season","Agricultural season","Categorical","Season","A, B or C"),
    ("province","Province","Categorical","Location","Rwandan province label"),
    ("district","District","Categorical","Location","District name"),
    ("altitude_m","Farm altitude","Numeric","Metres","Altitude above sea level"),
    ("rainfall_mm","Seasonal rainfall","Numeric","Millimetres","May contain missing values"),
    ("avg_temp_c","Average seasonal temperature","Numeric","Degrees Celsius","Average temperature"),
    ("soil_ph","Soil acidity/alkalinity","Numeric","pH","May contain missing values"),
    ("soil_type","Soil category","Categorical","Category","Observed soil classes"),
    ("seed_variety","Maize seed variety","Categorical","Category","Observed variety labels"),
    ("farming_practice","Farming practice type","Categorical","Category","Improved or Traditional"),
    ("npk_kg_ha","NPK fertiliser applied per hectare","Numeric","kg/ha","Rate per hectare"),
    ("urea_kg_ha","Urea fertiliser applied per hectare","Numeric","kg/ha","Rate per hectare"),
    ("manure_t_ha","Manure applied per hectare","Numeric","tonnes/ha","Rate per hectare"),
    ("plant_density_per_ha","Plant population per hectare","Integer","Plants/ha","Plant density"),
    ("farm_area_ha","Farm area","Numeric","Hectares","Farm size"),
    ("irrigation","Whether irrigation was used","Categorical","Yes/No","Binary category"),
    ("pest_incidence_index","Pest incidence index","Numeric","Index","Index scale needs source confirmation"),
    ("yield_t_ha","Maize yield per hectare — target","Numeric","tonnes/ha","Target for later modelling; not a predictor"),
], columns=["column","meaning","data_type","unit","allowed_values_or_notes"])
display(dictionary)
dictionary.to_csv(PROJECT/"docs"/"data_dictionary.csv", index=False)

,column,meaning,data_type,unit,allowed_values_or_notes
0,record_id,Unique record code,Categorical,Identifier,Unique code per record
1,year,Observation year,Integer,Year,2015–2024 in uploaded file
2,season,Agricultural season,Categorical,Season,"A, B or C"
3,province,Province,Categorical,Location,Rwandan province label
4,district,District,Categorical,Location,District name
5,altitude_m,Farm altitude,Numeric,Metres,Altitude above sea level
6,rainfall_mm,Seasonal rainfall,Numeric,Millimetres,May contain missing values
7,avg_temp_c,Average seasonal temperature,Numeric,Degrees Celsius,Average temperature
8,soil_ph,Soil acidity/alkalinity,Numeric,pH,May contain missing values
9,soil_type,Soil category,Categorical,Category,Observed soil classes


## Source log
Q1 file is supplied with the task.

In [7]:
source_log = pd.DataFrame([{
    "source": "CSV supplied by the lecturer/user with the assignment",
    "file": "maize_yield_rwanda_rw.csv",
    "date_accessed": pd.Timestamp.today().date().isoformat(),
    "licence_or_permission": "Coursework use; confirm any redistribution restrictions with lecturer",
    "collector": "Not specified in the uploaded CSV",
    "limitations": "Dataset metadata/source documentation not included with the CSV",
    "ethical_notes": "No personal names or direct personal identifiers observed; record_id is retained only for traceability"
}])
display(source_log)
source_log.to_csv(PROJECT/"docs"/"source_log.csv", index=False)

,source,file,date_accessed,licence_or_permission,collector,limitations,ethical_notes
0,CSV supplied by the lecturer/user with the ass...,maize_yield_rwanda_rw.csv,2026-11-12,Coursework use; confirm any redistribution res...,Not specified in the uploaded CSV,Dataset metadata/source documentation not incl...,No personal names or direct personal identifie...
